# Cleanup (OPTIONAL)

|Previous Notebook|Current Notebook|Next Notebook|
|:-:|:-:|:-:|
|[4.alignment.ipynb](./4.alignment.ipynb)|<i>5.cleanup.ipynb</i>|-|

This notebook clears up some last-minute dregs in the data. This notebook is _technically_ **OPTIONAL**. Depends on what kind of data you have.

In [1]:
import pandas as pd
import os

pd.set_option('display.max_colwidth', None)   # Show full cell contents
pd.set_option('display.max_columns', None)    # Show all columns
pd.set_option('display.width', None)          # Don't wrap lines
pd.set_option('display.float_format', '{:.0f}'.format)

## Cleaning up EEG Data

The EEG data was only recorded using `unix_ms` (technically it also has the Timestamp variable, but that's besides the point). In any case, we should at some point associate each `unix_ms` in the eeg data to a frame in our VR data. We can do this via a `merge_asof()`:

In [3]:
valid_participants = pd.read_csv('./data/valid_participant_offsets.csv')
for _, row in valid_participants.iterrows():

    # calcualte the participant directory
    pdir = f"./data/{row['p_id']}/trials/"
    print(pdir)
    
    # Get the offset for this participant
    offset = row['offset']

    # Get the simulation trials
    sims = pd.read_csv(os.path.join(pdir, 'simulations.csv')).drop(columns=['Unnamed: 0'])

    # Look at each trial
    for _, trial in sims.iterrows():
        trial_id = trial['trial_id']
        tdir = os.path.join(pdir, f"{trial_id}/simulation/")

        # Get the eeg processed and eeg_raw. Also get `eye.csv` to get a reference to both `unix_ms` and `frame` data.
        eeg_processed = pd.read_csv(tdir+'eeg_processed.csv').drop(columns=['Unnamed: 0'])
        eeg_raw = pd.read_csv(tdir+'eeg_raw.csv').drop(columns=['Unnamed: 0'])
        eye = pd.read_csv(tdir+'eye.csv')

        # Firstly, correct the timing
        eeg_processed['unix_ms_corrected'] = eeg_processed['unix_ms'] - offset
        eeg_raw['unix_ms_corrected'] = eeg_raw['unix_ms'] - offset

        # Secondly, make sure that both columns are integers (technically long)
        #eeg_processed['unix_ms_corrected'] = eeg_processed['unix_ms_corrected'].astype('int64')
        eeg_processed['unix_ms_corrected_int64'] = eeg_processed['unix_ms_corrected'].astype('int64')

        num_duplicates = eeg_processed.duplicated().sum()
        print(num_duplicates)

        eeg_processed_2 = eeg_processed.drop_duplicates()
        dt = eeg_processed_2['unix_ms'].diff()
        # Compute mean delta
        mean_dt_ms = dt.mean()
        # Convert to Hz (samples per second)
        sample_rate_hz = 1000 / mean_dt_ms
        print(f"Average sample rate: {sample_rate_hz:.2f} Hz")

        display(eeg_processed_2)
        

        """
        # Secondly, attach a `frame` column based on `eye`
        eeg_processed_corrected = pd.merge_asof(
            eeg_processed,
            eye[['unix_ms', 'frame']],
            left_on = 'unix_ms',
            right_on = 'unix_ms',
            direction = 'backward'
        )

        # Likely that some columns will be 

        display(eye, eeg_processed_corrected.tail(20))
        """



./data/P4_Muse2/trials/
5613
Average sample rate: 75.78 Hz


,unix_sec,unix_ms,rel_sec,rel_ms,Delta_TP9,Delta_TP10,Delta_AF7,Delta_AF8,Theta_TP9,Theta_TP10,Theta_AF7,Theta_AF8,Alpha_TP9,Alpha_TP10,Alpha_AF7,Alpha_AF8,Beta_TP9,Beta_TP10,Beta_AF7,Beta_AF8,Gamma_TP9,Gamma_TP10,Gamma_AF7,Gamma_AF8,unix_ms_corrected,unix_ms_corrected_int64
0,1759611096,1759611096188,32,32231,1,0,0,0,0,-0,-0,0,0,0,0,0,1,1,0,0,1,1,-0,0,1759611096135,1759611096135
5,1759611096,1759611096189,32,32232,1,0,0,0,0,-0,-0,0,0,0,0,0,1,1,0,0,1,1,-0,0,1759611096136,1759611096136
10,1759611096,1759611096190,32,32233,1,0,0,0,0,-0,-0,0,0,0,0,0,1,1,0,0,1,1,-0,0,1759611096137,1759611096137
12,1759611096,1759611096211,32,32254,1,0,0,0,0,-0,-0,0,0,0,0,0,1,1,0,0,1,1,-0,0,1759611096158,1759611096158
13,1759611096,1759611096212,32,32255,1,0,0,0,0,-0,-0,0,0,0,0,0,1,1,0,0,1,1,-0,0,1759611096159,1759611096159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7945,1759611127,1759611127249,63,63292,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1759611127196,1759611127196
7946,1759611127,1759611127250,63,63293,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1759611127197,1759611127197
7951,1759611127,1759611127251,63,63294,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1759611127198,1759611127198
7956,1759611127,1759611127252,63,63295,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1759611127199,1759611127199


4611
Average sample rate: 74.95 Hz


,unix_sec,unix_ms,rel_sec,rel_ms,Delta_TP9,Delta_TP10,Delta_AF7,Delta_AF8,Theta_TP9,Theta_TP10,Theta_AF7,Theta_AF8,Alpha_TP9,Alpha_TP10,Alpha_AF7,Alpha_AF8,Beta_TP9,Beta_TP10,Beta_AF7,Beta_AF8,Gamma_TP9,Gamma_TP10,Gamma_AF7,Gamma_AF8,unix_ms_corrected,unix_ms_corrected_int64
0,1759611127,1759611127297,63,63340,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1759611127244,1759611127244
4,1759611127,1759611127298,63,63341,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1759611127245,1759611127245
5,1759611127,1759611127299,63,63342,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1759611127246,1759611127246
11,1759611127,1759611127300,63,63343,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1759611127247,1759611127247
12,1759611127,1759611127341,63,63384,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1,1,-0,0,1759611127288,1759611127288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6499,1759611153,1759611152652,89,88695,-0,0,1,0,-0,0,0,0,0,1,1,0,1,1,0,0,1,1,-0,0,1759611152599,1759611152599
6503,1759611153,1759611152653,89,88696,-0,0,1,0,-0,0,0,0,0,1,1,0,1,1,0,0,1,1,-0,0,1759611152600,1759611152600
6504,1759611153,1759611152697,89,88740,-0,0,1,0,-0,0,0,0,0,1,1,0,1,1,0,0,1,1,-0,0,1759611152644,1759611152644
6509,1759611153,1759611152698,89,88741,-0,0,1,0,-0,0,0,0,0,1,1,0,1,1,0,0,1,1,-0,0,1759611152645,1759611152645


4811
Average sample rate: 79.08 Hz


...

,unix_sec,unix_ms,rel_sec,rel_ms,Delta_TP9,Delta_TP10,Delta_AF7,Delta_AF8,Theta_TP9,Theta_TP10,Theta_AF7,Theta_AF8,Alpha_TP9,Alpha_TP10,Alpha_AF7,Alpha_AF8,Beta_TP9,Beta_TP10,Beta_AF7,Beta_AF8,Gamma_TP9,Gamma_TP10,Gamma_AF7,Gamma_AF8,unix_ms_corrected,unix_ms_corrected_int64
0,1758754781,1758754780677,122,121596,1,0,-0,0,2,0,-0,0,2,0,0,0,2,0,1,0,1,0,1,0,1758754780577,1758754780577
6,1758754781,1758754780678,122,121597,1,0,-0,0,2,0,-0,0,2,0,0,0,2,0,1,0,1,0,1,0,1758754780578,1758754780578
11,1758754781,1758754780678,122,121597,1,0,-0,0,2,0,-0,0,2,0,0,0,2,0,1,0,1,0,1,0,1758754780578,1758754780578
12,1758754781,1758754780721,122,121640,1,0,-0,0,2,0,-0,0,2,0,0,0,2,0,1,0,1,0,1,0,1758754780621,1758754780621
14,1758754781,1758754780722,122,121641,1,0,-0,0,2,0,-0,0,2,0,0,0,2,0,1,0,1,0,1,0,1758754780622,1758754780622
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5844,1758754804,1758754803515,144,144434,1,1,-0,1,2,1,1,0,2,1,1,1,2,1,1,0,1,0,1,-0,1758754803415,1758754803415
5854,1758754804,1758754803516,144,144435,1,1,-0,1,2,1,1,0,2,1,1,1,2,1,1,0,1,0,1,-0,1758754803416,1758754803416
5856,1758754804,1758754803580,144,144499,1,1,-0,1,2,1,1,0,2,1,1,1,2,1,1,0,1,0,1,-0,1758754803480,1758754803480
5861,1758754804,1758754803580,144,144499,1,1,-0,1,2,1,1,0,2,1,1,1,2,1,1,0,1,0,1,-0,1758754803480,1758754803480


4923
Average sample rate: 64.79 Hz


,unix_sec,unix_ms,rel_sec,rel_ms,Delta_TP9,Delta_TP10,Delta_AF7,Delta_AF8,Theta_TP9,Theta_TP10,Theta_AF7,Theta_AF8,Alpha_TP9,Alpha_TP10,Alpha_AF7,Alpha_AF8,Beta_TP9,Beta_TP10,Beta_AF7,Beta_AF8,Gamma_TP9,Gamma_TP10,Gamma_AF7,Gamma_AF8,unix_ms_corrected,unix_ms_corrected_int64
0,1758754804,1758754803623,145,144542,1,1,-0,1,2,1,1,0,2,1,1,1,2,1,1,0,1,0,1,-0,1758754803523,1758754803523
1,1758754804,1758754803624,145,144543,1,1,-0,1,2,1,1,0,2,1,1,1,2,1,1,0,1,0,1,-0,1758754803524,1758754803524
9,1758754804,1758754803625,145,144544,1,1,-0,1,2,1,1,0,2,1,1,1,2,1,1,0,1,0,1,-0,1758754803525,1758754803525
12,1758754804,1758754803710,145,144629,1,1,-0,1,2,1,1,0,2,1,1,1,2,1,1,0,1,0,1,-0,1758754803610,1758754803610
16,1758754804,1758754803711,145,144630,1,1,-0,1,2,1,1,0,2,1,1,1,2,1,1,0,1,0,1,-0,1758754803611,1758754803611
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6567,1758754829,1758754829260,170,170179,1,1,0,0,2,0,0,-0,2,-0,1,0,2,1,1,0,1,0,1,-0,1758754829160,1758754829160
6571,1758754829,1758754829260,170,170179,1,1,0,0,2,-0,0,-0,2,0,1,0,2,0,1,0,1,0,1,-0,1758754829160,1758754829160
6572,1758754829,1758754829261,170,170180,1,1,0,0,2,-0,0,-0,2,0,1,0,2,0,1,0,1,0,1,-0,1758754829161,1758754829161
6576,1758754829,1758754829303,170,170222,1,1,0,0,2,-0,0,-0,2,0,1,0,2,0,1,0,1,0,1,-0,1758754829203,1758754829203


2459
Average sample rate: 66.08 Hz


,unix_sec,unix_ms,rel_sec,rel_ms,Delta_TP9,Delta_TP10,Delta_AF7,Delta_AF8,Theta_TP9,Theta_TP10,Theta_AF7,Theta_AF8,Alpha_TP9,Alpha_TP10,Alpha_AF7,Alpha_AF8,Beta_TP9,Beta_TP10,Beta_AF7,Beta_AF8,Gamma_TP9,Gamma_TP10,Gamma_AF7,Gamma_AF8,unix_ms_corrected,unix_ms_corrected_int64
0,1758754829,1758754829347,170,170266,1,1,0,0,2,-0,0,-0,2,0,1,0,2,0,1,0,1,0,1,-0,1758754829247,1758754829247
6,1758754829,1758754829348,170,170267,1,1,0,0,2,-0,0,-0,2,0,1,0,2,0,1,0,1,0,1,-0,1758754829248,1758754829248
9,1758754829,1758754829348,170,170267,1,1,0,0,2,-0,0,-0,2,0,0,0,2,0,1,0,1,0,1,-0,1758754829248,1758754829248
11,1758754829,1758754829349,170,170268,1,1,0,0,2,-0,0,-0,2,0,0,0,2,0,1,0,1,0,1,-0,1758754829249,1758754829249
12,1758754829,1758754829390,170,170309,1,1,0,0,2,-0,0,-0,2,0,0,0,2,0,1,0,1,0,1,-0,1758754829290,1758754829290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3288,1758754842,1758754842195,183,183114,2,2,0,1,2,2,0,-0,2,1,0,-0,1,1,1,0,1,0,1,-0,1758754842095,1758754842095
3294,1758754842,1758754842196,183,183115,2,2,0,1,2,2,0,-0,2,1,0,-0,1,1,1,0,1,0,1,-0,1758754842096,1758754842096
3300,1758754842,1758754842239,183,183158,2,2,0,1,2,2,0,-0,2,1,0,-0,1,1,1,0,1,0,1,-0,1758754842139,1758754842139
3306,1758754842,1758754842240,183,183159,2,2,0,1,2,2,0,-0,2,1,0,-0,1,1,1,0,1,0,1,-0,1758754842140,1758754842140
